# AirSight Time-Based Data Split

This notebook prepares a shared time-based data split for all forecasting models.

It covers:

1. Loading and sorting the model-ready traffic data by time.
2. Splitting the data into approximately 70% training, 15% validation, and 15% test sets.
3. Leaving three boundary rows between the sets to prevent the three-hour forecast targets from crossing into the next period.
4. Validating the time order and split boundaries.

The same split will be used by XGBoost, LSTM, and GRU to ensure a fair comparison.

## 1. Load the Model-Ready Data

The traffic dataset is loaded and sorted by time before splitting.

In [1]:
import pandas as pd
import numpy as np

traffic_data = pd.read_csv(
    'model_ready_traffic.csv',
    parse_dates=['hour']
)

traffic_data = (
    traffic_data
    .sort_values('hour')
    .reset_index(drop=True)
)

print("Dataset shape:", traffic_data.shape)
print("Start time:", traffic_data['hour'].min())
print("End time:", traffic_data['hour'].max())
print("Time is sorted:", traffic_data['hour'].is_monotonic_increasing)

display(traffic_data.head())

Dataset shape: (4857, 31)
Start time: 2025-03-21 21:00:00+00:00
End time: 2025-10-10 05:00:00+00:00
Time is sorted: True


,hour,flight_count,arrival,departure,terminal_1_count,terminal_2_count,terminal_3_count,terminal_4_count,terminal_5_count,unique_airlines,...,lag_3h,lag_24h,lag_48h,lag_168h,rolling_mean_3h,rolling_mean_24h,rolling_std_24h,target_t1,target_t2,target_t3
0,2025-03-21 21:00:00+00:00,26,14,12,5,2,1,4,14,10,...,31.0,25.0,24.0,26.0,34.000000,25.708333,4.804700,16.0,22.0,27.0
1,2025-03-21 22:00:00+00:00,16,9,7,1,0,3,0,12,7,...,36.0,20.0,20.0,19.0,32.333333,25.750000,4.802626,22.0,27.0,17.0
2,2025-03-21 23:00:00+00:00,22,10,12,2,0,4,6,10,9,...,35.0,20.0,20.0,24.0,25.666667,25.583333,5.072660,27.0,17.0,20.0
3,2025-03-22 00:00:00+00:00,27,18,9,5,0,7,4,11,11,...,26.0,23.0,24.0,22.0,21.333333,25.666667,4.992748,17.0,20.0,25.0
4,2025-03-22 01:00:00+00:00,17,7,10,0,0,5,1,11,8,...,16.0,19.0,20.0,16.0,21.666667,25.833333,4.966555,20.0,25.0,20.0


## 2. Split the Data by Time

The data is divided into training, validation, and test periods without shuffling.

Three boundary rows are left before the validation and test periods because the forecast horizon is three hours.

In [2]:
# Define the split points
total_rows = len(traffic_data)

train_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

# Maximum forecast horizon: 3 hours ahead
forecast_horizon = 3

# Split the data by time without shuffling
# Leave 3 boundary rows before validation and test
train_data = traffic_data.iloc[
    :train_end - forecast_horizon
].copy()

validation_data = traffic_data.iloc[
    train_end:validation_end - forecast_horizon
].copy()

test_data = traffic_data.iloc[
    validation_end:
].copy()

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Test rows:", len(test_data))

print("\nTraining period:")
print(train_data['hour'].min(), "to", train_data['hour'].max())

print("\nValidation period:")
print(validation_data['hour'].min(), "to", validation_data['hour'].max())

print("\nTest period:")
print(test_data['hour'].min(), "to", test_data['hour'].max())

Training rows: 3396
Validation rows: 726
Test rows: 729

Training period:
2025-03-21 21:00:00+00:00 to 2025-08-10 08:00:00+00:00

Validation period:
2025-08-10 12:00:00+00:00 to 2025-09-09 17:00:00+00:00

Test period:
2025-09-09 21:00:00+00:00 to 2025-10-10 05:00:00+00:00


### Validate the Split

The following checks confirm that the time order is preserved and the three-hour forecast targets do not cross the split boundaries.

In [3]:
# Six boundary rows are intentionally unused
unused_rows = forecast_horizon * 2

# Check the expected number of used rows
assert (
    len(train_data)
    + len(validation_data)
    + len(test_data)
    == len(traffic_data) - unused_rows
)

# Check that each set is sorted by time
assert train_data['hour'].is_monotonic_increasing
assert validation_data['hour'].is_monotonic_increasing
assert test_data['hour'].is_monotonic_increasing

# Check that the 3-hour forecast horizon
# does not cross the split boundaries
assert (
    train_data.index.max() + forecast_horizon
    < validation_data.index.min()
)

assert (
    validation_data.index.max() + forecast_horizon
    < test_data.index.min()
)

print("All split checks passed.")

All split checks passed.
